# SWaT R02 — Online-Retraining Aggregation

Reads `$SCRATCH/swat_paper_run/checkpoints/online/` (6,930 round JSONs) and produces:

| Output | Purpose |
|---|---|
| `online_results.csv` | long-format DataFrame, one row per round |
| `Table_X_per_seed_minima.csv` | 55 rows (11 detectors × 5 seeds), worst trajectory per (det, seed) |
| `Table_XI_per_config_summary.csv` | 18 rows (2 generators × 3 T × 3 Δp) |
| `Table_XII_gradual_vs_oneshot.csv` | matched-cumulative-budget comparison |
| `figures/Fig10_per_seed_trajectories.png` | F1 vs round, AE+PCA × 2 generators |
| `figures/Fig11_dF1_dFNR_scatter.png` | failure-mode dichotomy at peak |
| `figures/Fig12_gradual_vs_oneshot.png` | bars at 3% / 5% / 10% budgets |

**How to run on Narval:**

```bash
cd ~/projects/def-liyang/$USER/narval_swat_run
source venv/bin/activate
# Option A: launch a notebook server on a login node and tunnel it back to your Mac
jupyter notebook --no-browser --port=8888
# Then on your Mac in another terminal:
#   ssh -N -L 8888:localhost:8888 $USER@narval.alliancecan.ca
# And open http://localhost:8888 in your browser; navigate to notebook/SWaT_R02_Aggregator.ipynb

# Option B: open the notebook in VS Code Remote-SSH and run cells normally.
```

After running every cell, rsync the outputs back to your Mac — see the last cell.

In [ ]:
# 0. Imports + path setup (robust to running from project root or notebook/ subfolder)
import os, sys, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

# Find project root: either current dir contains src/, or its parent does.
_cwd = Path.cwd()
if (_cwd / 'src').is_dir():
    PROJECT_ROOT = _cwd
elif (_cwd.parent / 'src').is_dir():
    PROJECT_ROOT = _cwd.parent
else:
    raise RuntimeError(f'Could not find src/ from cwd={_cwd}. cd to narval_swat_run/ first.')
sys.path.insert(0, str(PROJECT_ROOT))
print(f'PROJECT_ROOT = {PROJECT_ROOT}')

from src.config import CONFIG, get_output_dir, online_combos
from src.attacks_online import trajectory_filename

OUT_BASE   = get_output_dir()
ONLINE_DIR = OUT_BASE / 'checkpoints' / 'online'
ATTACK_DIR = OUT_BASE / 'checkpoints' / 'attacks'
AGG_DIR    = OUT_BASE / 'online_aggregated'
FIG_DIR    = AGG_DIR / 'figures'
AGG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'Reading from : {ONLINE_DIR}')
print(f'Writing to   : {AGG_DIR}')

## 1. Load all online JSONs into a long-format DataFrame

One row per (detector, generator, T, Δp, seed, round). Saves to `online_results.csv`.

In [ ]:
DETECTOR_FAMILY = {
    'iforest':     'tree',
    'svm':         'kernel',
    'lof':         'local_density',
    'cluster':     'cluster_density',
    'knn':         'distance',
    'histogram':   'statistical',
    'pca':         'subspace',
    'mcd':         'robust_covariance',
    'abod':        'geometric',
    'autoencoder': 'neural_pointwise',
    'lstm_ae':     'neural_sequence',
}

def safe_load(fp):
    try:
        with open(fp) as f:
            return json.load(f)
    except Exception as e:
        print(f'WARN: failed {fp.name}: {e}')
        return None

files = [f for f in os.listdir(ONLINE_DIR)
         if f.endswith('.json') and not f.endswith('.error.json')]
print(f'Found {len(files)} round JSONs (expect 6930)')

rows = [safe_load(ONLINE_DIR / f) for f in files]
rows = [r for r in rows if r is not None]
long_df = pd.DataFrame(rows)
long_df['family'] = long_df['model'].map(DETECTOR_FAMILY)
long_df = long_df.sort_values(
    ['model', 'generator', 'T', 'delta_p', 'seed', 'round']
).reset_index(drop=True)

long_path = AGG_DIR / 'online_results.csv'
long_df.to_csv(long_path, index=False)
print(f'Saved {long_path} — {len(long_df)} rows, {len(long_df.columns)} columns')
long_df.head()

## 2. Table X — Per-(detector, seed) trajectory minima

For each (detector, seed), find the trajectory (across all 18 generator/T/Δp configs) with the lowest min-F1, and report clean F1, min F1, ΔF1, peak round, peak cumulative budget, FNR at peak, and ΔFNR at peak.

**Expected: 55 rows = 11 detectors × 5 seeds.**

In [ ]:
def build_table_x(df):
    rows = []
    for (model, seed), g in df.groupby(['model', 'seed']):
        traj_summaries = []
        for (gen, T, dp), tg in g.groupby(['generator', 'T', 'delta_p']):
            tg = tg.sort_values('round')
            clean = tg[tg['round'] == 0]
            non_clean = tg[tg['round'] > 0]
            if clean.empty or non_clean.empty:
                continue
            cf1 = float(clean['f1'].iloc[0])
            cfnr = float(clean['fnr'].iloc[0])
            peak = non_clean.loc[non_clean['f1'].idxmin()]
            traj_summaries.append({
                'generator': gen, 'T': int(T), 'delta_p': float(dp),
                'clean_f1': cf1, 'clean_fnr': cfnr,
                'min_f1': float(peak['f1']),
                'delta_f1': float(peak['f1']) - cf1,
                'peak_round': int(peak['round']),
                'peak_cumulative_p': float(peak['cumulative_p']),
                'fnr_at_peak': float(peak['fnr']),
                'delta_fnr_at_peak': float(peak['fnr']) - cfnr,
            })
        if not traj_summaries:
            continue
        worst = min(traj_summaries, key=lambda r: r['min_f1'])
        rows.append({
            'model': model, 'seed': int(seed),
            'family': DETECTOR_FAMILY.get(model, ''),
            'clean_f1': worst['clean_f1'],
            'min_f1': worst['min_f1'],
            'delta_f1': worst['delta_f1'],
            'peak_round': worst['peak_round'],
            'peak_cumulative_p': worst['peak_cumulative_p'],
            'fnr_at_peak': worst['fnr_at_peak'],
            'delta_fnr_at_peak': worst['delta_fnr_at_peak'],
            'worst_generator': worst['generator'],
            'worst_T': worst['T'],
            'worst_dp': worst['delta_p'],
        })
    return pd.DataFrame(rows).sort_values(['model', 'seed']).reset_index(drop=True)

table_x = build_table_x(long_df)
table_x.to_csv(AGG_DIR / 'Table_X_per_seed_minima.csv', index=False)
print(f'Table X: {len(table_x)} rows (expect 55)')
table_x

## 3. Table XI — Per-(generator, T, Δp) summary

For each of the 18 cells, median per-seed min F1 + IQR + mean ΔFNR at peak. Aggregates across 11 detectors × 5 seeds = 55 trajectories per cell.

**Expected: 18 rows.**

In [ ]:
def build_table_xi(df):
    rows = []
    for (gen, T, dp), g in df.groupby(['generator', 'T', 'delta_p']):
        traj_minima = []
        for (model, seed), tg in g.groupby(['model', 'seed']):
            tg = tg.sort_values('round')
            clean = tg[tg['round'] == 0]
            non_clean = tg[tg['round'] > 0]
            if clean.empty or non_clean.empty:
                continue
            cf1 = float(clean['f1'].iloc[0])
            cfnr = float(clean['fnr'].iloc[0])
            peak = non_clean.loc[non_clean['f1'].idxmin()]
            traj_minima.append({
                'min_f1': float(peak['f1']),
                'delta_f1': float(peak['f1']) - cf1,
                'delta_fnr_at_peak': float(peak['fnr']) - cfnr,
            })
        if not traj_minima:
            continue
        df_t = pd.DataFrame(traj_minima)
        rows.append({
            'generator': gen, 'T': int(T), 'delta_p': float(dp),
            'n_trajectories': len(df_t),
            'median_min_f1': float(df_t['min_f1'].median()),
            'iqr_min_f1':    float(df_t['min_f1'].quantile(0.75) - df_t['min_f1'].quantile(0.25)),
            'min_min_f1':    float(df_t['min_f1'].min()),
            'max_min_f1':    float(df_t['min_f1'].max()),
            'mean_delta_f1': float(df_t['delta_f1'].mean()),
            'mean_delta_fnr':float(df_t['delta_fnr_at_peak'].mean()),
        })
    return pd.DataFrame(rows).sort_values(['generator', 'T', 'delta_p']).reset_index(drop=True)

table_xi = build_table_xi(long_df)
table_xi.to_csv(AGG_DIR / 'Table_XI_per_config_summary.csv', index=False)
print(f'Table XI: {len(table_xi)} rows (expect 18)')
table_xi

## 4. Table XII — Gradual vs one-shot at matched cumulative budgets

Pulls one-shot baseline from `checkpoints/attacks/random_flip/` (the existing main grid). Compares median min F1 across (detector, seed) for matched budgets {3%, 5%, 10%}.

In [ ]:
matched_budgets = [0.03, 0.05, 0.10]
tol = 0.005   # tolerance for matching cumulative_p ≈ budget

# ---- one-shot from existing baseline (random_flip) ----
rf_dir = ATTACK_DIR / 'random_flip'
oneshot = {b: [] for b in matched_budgets}
if rf_dir.exists():
    for f in sorted(os.listdir(rf_dir)):
        if not f.endswith('.json'):
            continue
        d = safe_load(rf_dir / f)
        if d is None or 'f1' not in d:
            continue
        rate = float(d.get('poison_rate', -1))
        for b in matched_budgets:
            if abs(rate - b) < 1e-6:
                oneshot[b].append(float(d['f1']))
                break
else:
    print(f'WARN: {rf_dir} not found — one-shot column will be NaN')

# ---- gradual: per-(model, seed) min F1 within ±tol of each matched budget ----
xii_rows = []
for b in matched_budgets:
    mask = np.isclose(long_df['cumulative_p'], b, atol=tol)
    gradual_rows = long_df[mask]
    per_ms = gradual_rows.groupby(['model', 'seed'])['f1'].min().reset_index()
    g_f1s = per_ms['f1'].values
    o_f1s = oneshot.get(b, [])
    xii_rows.append({
        'cumulative_p':         b,
        'n_gradual':            len(g_f1s),
        'median_gradual_f1':    float(np.median(g_f1s)) if len(g_f1s) else float('nan'),
        'iqr_gradual_f1':       float(np.subtract(*np.percentile(g_f1s, [75, 25]))) if len(g_f1s) else float('nan'),
        'n_oneshot':            len(o_f1s),
        'median_oneshot_f1':    float(np.median(o_f1s)) if len(o_f1s) else float('nan'),
        'iqr_oneshot_f1':       float(np.subtract(*np.percentile(o_f1s, [75, 25]))) if len(o_f1s) else float('nan'),
        'gradual_minus_oneshot': (float(np.median(g_f1s)) - float(np.median(o_f1s)))
                                 if (len(g_f1s) and len(o_f1s)) else float('nan'),
    })

table_xii = pd.DataFrame(xii_rows)
table_xii.to_csv(AGG_DIR / 'Table_XII_gradual_vs_oneshot.csv', index=False)
print(f'Table XII: {len(table_xii)} rows')
table_xii

## 5. Headline numbers

These are the numbers you'll quote in the abstract and §VI-B.2 narrative. The pilot predicted AE per-seed mean min F1 ≈ 0.77 with ΔF1 ≈ −0.16.

In [ ]:
ae_x = table_x[table_x['model'] == 'autoencoder']
print('=== HEADLINE NUMBERS ===')
if len(ae_x):
    print(f"AE per-seed min F1 (worst config): mean={ae_x['min_f1'].mean():.4f}, "
          f"clean={ae_x['clean_f1'].mean():.4f}, "
          f"ΔF1={ae_x['delta_f1'].mean():+.4f}")
print(f"All-detector mean per-seed min F1: {table_x['min_f1'].mean():.4f}")
print(f"All-detector mean ΔF1:             {table_x['delta_f1'].mean():+.4f}")
print(f"All-detector mean ΔFNR at peak:    {table_x['delta_fnr_at_peak'].mean():+.4f}")
print()
print('Per-detector mean (sorted by ΔF1, most-damaged first):')
(table_x.groupby('model')[['clean_f1', 'min_f1', 'delta_f1', 'delta_fnr_at_peak']]
         .mean()
         .sort_values('delta_f1'))

## 6. Figure 10 — Per-seed trajectories (paper headline figure)

F1 vs round, one line per seed, for AE and PCA × random_injection / high_loss. The × marker shows the peak-damage round per seed.

In [ ]:
def fig10_plot():
    detectors = ['autoencoder', 'pca']
    generators = ['random_injection', 'high_loss']
    fig, axes = plt.subplots(len(detectors), len(generators),
                              figsize=(11, 4 * len(detectors)), sharey=True)
    for i, det in enumerate(detectors):
        for j, gen in enumerate(generators):
            ax = axes[i][j]
            sub = long_df[(long_df['model'] == det) & (long_df['generator'] == gen)]
            seeds = sorted(sub['seed'].unique())
            cmap = plt.get_cmap('viridis', len(seeds))
            for s_i, s in enumerate(seeds):
                seed_data = sub[sub['seed'] == s]
                traj_groups = list(seed_data.groupby(['T', 'delta_p']))
                if not traj_groups:
                    continue
                worst_key, worst_df = min(traj_groups, key=lambda kv: kv[1]['f1'].min())
                worst_df = worst_df.sort_values('round')
                ax.plot(worst_df['round'], worst_df['f1'], marker='o', color=cmap(s_i),
                        label=f"seed={s} (T={worst_key[0]}, Δp={worst_key[1]})")
                peak = worst_df.loc[worst_df['f1'].idxmin()]
                ax.scatter([peak['round']], [peak['f1']], s=120, marker='x',
                            color=cmap(s_i), zorder=5)
            ax.set_title(f"{det} / {gen}")
            ax.set_xlabel('round')
            ax.set_ylabel('F1')
            ax.grid(True, alpha=0.3)
            ax.legend(fontsize=7, loc='lower left')
    plt.suptitle('Fig. 10  Per-seed trajectories (worst T/Δp per seed; × = peak damage)',
                  fontsize=12, y=1.02)
    plt.tight_layout()
    out_path = FIG_DIR / 'Fig10_per_seed_trajectories.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved {out_path}')
    plt.show()

fig10_plot()

## 7. Figure 11 — (ΔF1, ΔFNR) joint scatter at peak

One point per seed-trajectory at peak, colored by detector family. Bottom-right quadrant = combined precision+recall collapse; bottom-left = precision-only failure.

In [ ]:
def fig11_plot():
    fig, ax = plt.subplots(figsize=(8, 6))
    families = sorted(table_x['family'].dropna().unique())
    cmap = plt.get_cmap('tab20', len(families))
    fc = {f: cmap(i) for i, f in enumerate(families)}
    for fam in families:
        sub = table_x[table_x['family'] == fam]
        ax.scatter(sub['delta_f1'], sub['delta_fnr_at_peak'],
                    s=80, color=fc[fam], alpha=0.75,
                    label=fam, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.axvline(0, color='black', linestyle='--', linewidth=0.5, alpha=0.5)
    ax.set_xlabel('ΔF1 at peak (negative = damage)')
    ax.set_ylabel('ΔFNR at peak (positive = recall failure; negative = precision-only)')
    ax.set_title("Fig. 11  Joint (ΔF1, ΔFNR) scatter at each seed's peak")
    ax.grid(True, alpha=0.3)
    ax.legend(fontsize=8, loc='best', framealpha=0.9)
    plt.tight_layout()
    out_path = FIG_DIR / 'Fig11_dF1_dFNR_scatter.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved {out_path}')
    plt.show()

fig11_plot()

## 8. Figure 12 — Gradual vs one-shot at matched cumulative budgets

Side-by-side bars at {3%, 5%, 10%}. Smaller gradual median F1 than one-shot at the same cumulative budget = gradual poisoning is more damaging.

In [ ]:
def fig12_plot():
    fig, ax = plt.subplots(figsize=(7, 5))
    x = np.arange(len(table_xii))
    w = 0.35
    ax.bar(x - w/2, table_xii['median_oneshot_f1'], width=w,
            label='One-shot (random_flip baseline)', color='#888')
    ax.bar(x + w/2, table_xii['median_gradual_f1'], width=w,
            label='Gradual (online retraining)', color='#c44')
    ax.set_xticks(x)
    ax.set_xticklabels([f"{p:.0%}" for p in table_xii['cumulative_p']])
    ax.set_xlabel('cumulative poison budget')
    ax.set_ylabel('median F1 across (detector × seed)')
    ax.set_title('Fig. 12  Gradual vs one-shot at matched cumulative budgets')
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend(loc='lower left', fontsize=9)
    for i, row in table_xii.iterrows():
        gap = row['gradual_minus_oneshot']
        if not np.isnan(gap):
            y = max(row['median_oneshot_f1'], row['median_gradual_f1']) + 0.02
            ax.annotate(f"Δ={gap:+.3f}", (x[i], y), ha='center', fontsize=9)
    plt.tight_layout()
    out_path = FIG_DIR / 'Fig12_gradual_vs_oneshot.png'
    plt.savefig(out_path, dpi=150, bbox_inches='tight')
    print(f'Saved {out_path}')
    plt.show()

fig12_plot()

## 9. Done — rsync the outputs back to your Mac

From a Mac terminal:

```bash
rsync -av \
    $USER@narval.alliancecan.ca:$SCRATCH/swat_paper_run/online_aggregated/ \
    ./swat_paper_run/online_aggregated/
```

That folder contains all 4 CSVs and 3 PNGs (~5 MB total) — everything you need for Phase 2 of the paper writeup.